# 🎯 CLI 페르소나 챗봇 만들기


## 📋  흐름

 챗봇을 처음 만들 때 거치는 5단계를 그대로 따라갑니다.

```
Step 0. 초기 설정          ← 패키지 설치, API 키 설정 (Colab/로컬 공통)
Step 1. 환경 점검          ← 1·2교시 코드가 정상 동작하는지 확인
Step 2. 페르소나 설계       ← 1단계+2단계 (요구사항·system 메시지)
Step 3. 챗봇 함수 만들기    ← 3단계 (MVP, prompt | llm)
Step 4. 명령어 처리        ← 4단계 (인터랙션)
Step 5. CLI 대화 루프      ← 4단계 (완성)
Step 6. 시연 시나리오 검증  ← 5단계

```


---

# Step 0. 초기 설정

## 0-1. 패키지 설치

LangChain 관련 패키지를 설치합니다.

- `langchain` : LangChain 메인 패키지
- `langchain-openai` : OpenAI 모델 연동
- `python-dotenv` : `.env` 파일 로드용



In [1]:
# [문제 1] 패키지 설치 (Colab/로컬 공통)
# ------------------------------------------------------------
# 요구사항
#   - 노트북 매직 명령 %pip 를 사용해 아래 3개 패키지를 "조용히(-q)" 설치하세요.
#       1) langchain          : LangChain 메인 패키지
#       2) langchain-openai   : OpenAI 모델 연동
#       3) python-dotenv      : .env 파일 로드용
#   - 한 줄로 작성합니다.
# ▶ 출력 예시: (에러 없이 설치 완료)
# ------------------------------------------------------------

!pip install -q langchain langchain-openai python-dotenv

## 0-2. OpenAI API 키 설정




In [2]:
# [문제 2] 실행 환경 감지
# ------------------------------------------------------------
# 요구사항
#   - try / except ImportError 구문으로 google.colab 모듈 import 를 시도하세요.
#   - import 성공 시  → 변수 IS_COLAB = True  , "✅ Google Colab" 출력
#   - ImportError 시 → 변수 IS_COLAB = False , "✅ 로컬 Jupyter" 출력
#   - IS_COLAB 변수는 다음 문제에서 사용합니다.
# ▶ 출력 예시: ✅ Google Colab   (또는 ✅ 로컬 Jupyter)
# ------------------------------------------------------------

try:
    import google.colab
    IS_COLAB = True
    print("Google Colab")
except ImportError:
    IS_COLAB = False
    print("로컬 Jupyter")

로컬 Jupyter


In [ ]:
# [문제 3] OpenAI API 키 로드
# ------------------------------------------------------------
# 요구사항
#   - os 모듈을 import 하세요.
#   - IS_COLAB 이 True 이면:
#       google.colab 의 userdata 를 import 하고
#       userdata.get("OPENAI_API_KEY") 값을 os.environ["OPENAI_API_KEY"] 에 저장
#   - IS_COLAB 이 False 이면:
#       dotenv 의 load_dotenv 를 import 하여 호출 (.env 파일 자동 로드)
#   - 마지막에 os.environ.get("OPENAI_API_KEY") 존재 여부로
#       있으면 "✅ API 키 OK", 없으면 "❌ 키 없음" 을 출력 (조건부 표현식 1줄)
#   ⚠️ 키 문자열("sk-...")을 코드에 직접 쓰지 마세요.
# ▶ 출력 예시: ✅ API 키 OK
# ------------------------------------------------------------

import os

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("API 키 OK" if os.environ.get("OPENAI_API_KEY") else "키 없음")

API 키 OK


---

# Step 1. 환경 점검

## 1-1. 필요한 패키지 import

| `ChatOpenAI` | LangChain의 OpenAI 모델 클래스 
| `ChatPromptTemplate` | 프롬프트 템플릿 


In [5]:
# [문제 4] 패키지 import
# ------------------------------------------------------------
# 요구사항
#   - langchain_openai 에서 ChatOpenAI 를 import
#   - langchain_core.prompts 에서 ChatPromptTemplate 를 import
#   - "✅ import 완료" 를 출력
# ------------------------------------------------------------

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

print("import 완료")

import 완료


## 1-2. 기본 패턴이 동작하는지 확인


In [7]:
# [문제 5] 기본 패턴 prompt | llm 체인 만들기
# ------------------------------------------------------------
# 요구사항
#   (1) LLM 생성 → 변수 llm
#       - 모델: "gpt-4o-mini"
#       - temperature: 0.3
#   (2) 프롬프트 템플릿 생성 → 변수 prompt
#       - ChatPromptTemplate.from_messages([...]) 사용
#       - system 메시지: "당신은 {role}입니다."   ({role} 은 변수 자리)
#       - human  메시지: "{question}"
#   (3) LCEL 파이프 연산자( | )로 prompt 와 llm 을 연결 → 변수 chain
#   (4) chain.invoke() 에 딕셔너리를 넘겨 동작 확인
#       - role     : "데이터 분석가"
#       - question : "이상치란 무엇인가요? 한 문장으로."
#   (5) 결과 객체(AIMessage)의 .content 를 출력
#   ※ llm, prompt, chain 변수는 이후 문제에서 계속 사용합니다.
# ▶ 출력 예시: 이상치란 데이터 집합에서 다른 값들과 크게 벗어난 값을 말합니다.
# ------------------------------------------------------------

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 {role}입니다."),
        ("human", "{question}"),
    ]
)

chain = prompt | llm

result = chain.invoke({"role": "데이터 분석가", "question": "이상치란 무엇인가요? 한 문장으로."})

print(result.content)


이상치는 데이터 집합에서 다른 값들과 현저하게 차이나는 값으로, 일반적인 패턴이나 경향에서 벗어난 데이터를 의미합니다.


**✅ 결과 확인**

분석가 톤의 한 문장 답변이 출력되면 환경 OK입니다.




In [9]:
# [문제 6] 페르소나 사전 만들기
# ------------------------------------------------------------
# 요구사항
#   - 딕셔너리 변수 PERSONAS 를 만드세요.  key = 페르소나 이름, value = system 메시지
#   - 아래 5개 페르소나를 모두 등록합니다.
#       "분석가"   : 10년 차 데이터 분석가 / 수치와 근거를 들어 간결하게 답변
#       "마케터"   : 디지털 마케팅 전문가 / 비즈니스 관점과 사례를 들어 답변
#       "개발자"   : 시니어 백엔드 개발자 / 코드 예시와 함께 실용적으로 답변
#       "기획자"   : IT 서비스 기획자 / 사용자 가치와 우선순위 관점에서 답변
#       "디자이너" : UX 디자이너 / 사용자 경험과 시각적 비유로 쉽게 답변
#     (각 value 는 "당신은 ~입니다. ~하세요." 형태의 완전한 문장으로 작성)
#   - f-string 으로 등록 개수(len)와 key 목록(list)을 출력하세요.
# ▶ 출력 예시: ✅ 5개 페르소나 등록 완료: ['분석가', '마케터', '개발자', '기획자', '디자이너']
# ------------------------------------------------------------

PERSONAS = {
    "분석가"   : "10년 차 데이터 분석가 / 수치와 근거를 들어 간결하게 답변",
    "마케터"   : "디지털 마케팅 전문가 / 비즈니스 관점과 사례를 들어 답변",
    "개발자"   : "시니어 백엔드 개발자 / 코드 예시와 함께 실용적으로 답변",
    "기획자"   : "IT 서비스 기획자 / 사용자 가치와 우선순위 관점에서 답변",
    "디자이너" : "UX 디자이너 / 사용자 경험과 시각적 비유로 쉽게 답변",    
}

print(f"{len(PERSONAS)}개 페르소나 등록 완료: {list(PERSONAS.keys())}")

5개 페르소나 등록 완료: ['분석가', '마케터', '개발자', '기획자', '디자이너']


In [12]:
# [문제 7] 같은 질문, 다른 페르소나 비교
# ------------------------------------------------------------
# 요구사항
#   - 변수 question = "좋은 데이터란 무엇인가요?"
#   - for 문으로 ["분석가", "마케터"] 두 페르소나를 순회하며
#       chain.invoke({"role": <해당 페르소나의 system 메시지>, "question": question}) 호출
#       (system 메시지는 PERSONAS[이름] 으로 조회)
#   - 매 반복마다 다음 형식으로 출력
#       빈 줄 + [페르소나이름]
#       답변 내용(.content)
#       "-" 60개 구분선
# ▶ 출력 예시:
#   [분석가]
#   좋은 데이터란 정확성, 완전성 ... 
#   ------------------------------------------------------------
# ------------------------------------------------------------

question = "좋은 데이터란 무엇인가요?"

for name in ["분석가", "마케터"]:
    result = chain.invoke({"role": PERSONAS[name], "question": question})
    print(f"\n[{name}]\n{result.content}")
    print("-"*60)


[분석가]
좋은 데이터란 다음과 같은 특성을 갖춘 데이터를 의미합니다:

1. **정확성 (Accuracy)**: 데이터가 실제를 정확하게 반영해야 합니다. 오류나 왜곡이 없어야 합니다.

2. **완전성 (Completeness)**: 필요한 모든 정보가 포함되어 있어야 하며, 결측치가 최소화되어야 합니다.

3. **일관성 (Consistency)**: 데이터가 여러 출처나 시간에 걸쳐 일관되게 유지되어야 하며, 서로 충돌하지 않아야 합니다.

4. **적시성 (Timeliness)**: 데이터가 필요한 시점에 제공되어야 하며, 최신 정보를 반영해야 합니다.

5. **관련성 (Relevance)**: 특정 분석이나 의사결정에 필요한 데이터여야 하며, 불필요한 정보는 배제되어야 합니다.

6. **유용성 (Usability)**: 데이터가 쉽게 접근 가능하고, 이해하기 쉬워야 하며, 분석에 적합해야 합니다.

이러한 특성을 갖춘 데이터는 신뢰할 수 있는 분석 결과를 도출하는 데 필수적입니다.
------------------------------------------------------------

[마케터]
좋은 데이터란 비즈니스 의사결정, 마케팅 전략 수립, 고객 분석 등 다양한 분야에서 효과적으로 활용될 수 있는 데이터입니다. 좋은 데이터의 특성은 다음과 같습니다:

1. **정확성 (Accuracy)**: 데이터가 사실과 일치하고 오류가 없어야 합니다. 부정확한 데이터는 잘못된 결정을 초래할 수 있습니다.

2. **일관성 (Consistency)**: 데이터가 서로 다른 출처나 시스템에서 일관되게 유지되어야 합니다. 예를 들어, 고객의 이름이나 주소가 여러 데이터베이스에서 동일해야 합니다.

3. **완전성 (Completeness)**: 필요한 모든 데이터가 포함되어 있어야 합니다. 예를 들어, 고객의 구매 이력을 분석할 때 모든 거래 데이터가 포함되어야 정확한 분석이 가능합니다.

4. **적시성 (Timeliness)**: 데이터가 최신이

---

# Step 3. 챗봇 함수 만들기 

In [13]:
# [문제 8] 챗봇 핵심 함수 ask() 만들기
# ------------------------------------------------------------
# 요구사항
#   - 함수 시그니처: ask(persona_name: str, question: str) -> str
#   - 함수 내부 처리 순서
#       1. persona_name 으로 PERSONAS 에서 system 메시지를 조회 → system_message
#       2. chain.invoke() 에 {"role": system_message, "question": question} 전달
#       3. 결과(AIMessage)에서 .content 텍스트만 꺼내 return
#   - 이 셀은 함수 정의만 하고 출력은 없습니다.
# ------------------------------------------------------------

def ask(persona_name: str, question: str) -> str:
    system_message = PERSONAS[persona_name]
    result = chain.invoke({"role": system_message, "question": question})
    return result.content

## 3-2. 함수 테스트

만든 함수를 실제로 호출해봅시다.


In [15]:
# [문제 9] ask() 함수 테스트
# ------------------------------------------------------------
# 요구사항
#   - ask() 를 "분석가" 페르소나, 질문 "A/B 테스트를 한 줄로 설명해주세요" 로 호출
#   - 반환값을 변수 answer 에 저장하고 출력
# ▶ 출력 예시: A/B 테스트는 두 가지 버전을 비교해 더 나은 성과를 내는 쪽을 찾는 실험입니다.
# ------------------------------------------------------------

answer = ask("분석가", "A/B 테스트를 한 줄로 설명해주세요")
print(answer)

A/B 테스트는 두 가지 이상의 변수를 비교하여 어떤 것이 더 효과적인지를 통계적으로 분석하는 실험 방법입니다.


---

# Step 4. 명령어 처리 

## 4-1. 사용자 입력의 두 종류

CLI 챗봇 사용자는 두 가지를 입력합니다.

| 입력 | 의미 | 처리 방법 |
| --- | --- | --- |
| 일반 텍스트 | LLM에게 보낼 질문 | `ask()` 호출 |
| `/`로 시작하는 텍스트 | 명령어 | 별도 처리 |

명령어를 처리하는 함수를 따로 만들어봅시다. 챗봇 로직(`ask`)과 명령어 로직을 분리하면 코드가 깔끔해져요.

## 4-2. 명령어 처리 함수

이 함수는 다음 명령어를 처리합니다.

- `/quit`, `/exit` → 종료 신호 반환
- `/change 마케터` → 페르소나 변경
- `/persona` → 현재 페르소나 확인
- `/help` → 도움말


In [ ]:
# [문제 10] 명령어 처리 함수 handle_command() 만들기
# ------------------------------------------------------------
# 요구사항
#   - 함수 시그니처: handle_command(user_input: str, current_persona: str)
#   - 반환값: (페르소나이름, 종료여부) 튜플   예) ("분석가", False)
#
#   [입력 파싱]
#   - user_input.split(maxsplit=1) 로 명령어와 인자를 분리 → parts
#       예) "/change 마케터" → ["/change", "마케터"]
#   - 첫 요소를 cmd 로 사용
#
#   [분기 처리]
#   1) cmd 가 "/quit" 또는 "/exit"  (in 튜플 사용)
#        → "👋 종료합니다." 출력 후 (current_persona, True) 즉시 반환
#   2) cmd == "/persona"
#        → "🎭 현재 페르소나: <현재 페르소나>" 출력
#   3) cmd == "/help"
#        → "📖 명령어: /quit, /exit, /change <이름>, /persona, /help" 출력
#   4) cmd == "/change"
#        → 인자가 있으면 parts[1], 없으면 "" 을 new_name 으로
#        → new_name 이 PERSONAS 에 있으면
#             "✅ 페르소나가 '<new_name>'(으)로 변경되었습니다." 출력 후 (new_name, False) 반환
#        → 없으면
#             "❌ '<new_name>'은 등록된 페르소나가 아닙니다." 출력
#             "   사용 가능: 분석가, 마케터, ..." 출력 (", ".join 사용)
#   5) 그 외 명령어
#        → "❓ 알 수 없는 명령어: <cmd> (/help 입력)" 출력
#
#   - 페르소나가 바뀌지 않은 모든 경우 마지막에 (current_persona, False) 반환
# ------------------------------------------------------------

## 4-3. 명령어 처리 테스트

대화 루프를 만들기 전에 명령어가 잘 동작하는지 확인합시다.


In [ ]:
# [문제 11] 명령어 처리 테스트 (시뮬레이션)
# ------------------------------------------------------------
# 요구사항
#   - 변수 current = "분석가" 로 시작
#   - handle_command() 를 아래 순서로 5번 호출하고, 반환된 페르소나로 current 를 갱신
#     (종료여부 값은 사용하지 않으므로 _ 로 받기)
#       1) "/persona"
#       2) "/change 마케터"
#       3) "/persona"
#       4) "/change 외계인"     ← 잘못된 페르소나 (거절되어야 함)
#       5) "/help"
# ▶ 출력 예시:
#   🎭 현재 페르소나: 분석가
#   ✅ 페르소나가 '마케터'(으)로 변경되었습니다.
#   🎭 현재 페르소나: 마케터
#   ❌ '외계인'은 등록된 페르소나가 아닙니다.
#      사용 가능: 분석가, 마케터, 개발자, 기획자, 디자이너
#   📖 명령어: /quit, /exit, /change <이름>, /persona, /help
# ------------------------------------------------------------

**✅ 결과 확인**

- `/persona`로 현재 페르소나 확인됨
- `/change 마케터`로 변경됨
- `/change 외계인` 같은 잘못된 이름은 거절됨
- `/help`로 도움말이 출력됨

이 4가지가 모두 동작하면 명령어 처리는 완성!


---

# Step 5. CLI 대화 루프

## 5-1. 대화 루프란?

지금까지 만든 부품은 3개입니다.

```
PERSONAS          ← 페르소나 사전 (Step 2)
ask()             ← 챗봇 두뇌 (Step 3)
handle_command()  ← 명령어 처리 (Step 4)
```

이걸 모두 묶어서 **"입력을 기다리고 → 처리하고 → 다시 입력 기다리는"** 무한 루프를 만들면 챗봇이 완성됩니다.

## 5-2. 챗봇 실행 함수


In [ ]:
# [문제 12] 챗봇 실행 함수 run_chatbot() 만들기
# ------------------------------------------------------------
# 요구사항
#   - 함수 시그니처: run_chatbot(initial_persona: str = "분석가")
#   - 지역 변수 persona 에 initial_persona 를 저장
#   - 시작 안내 2줄 출력
#       "👋 페르소나 챗봇입니다. (현재: <persona>)"
#       "   /help 로 명령어 보기, /quit 로 종료"  + 빈 줄
#
#   [무한 루프 while True]
#   1) input("질문> ") 으로 입력받고 .strip() 으로 공백 제거 → user_input
#   2) 빈 입력이면 continue (무시)
#   3) "/" 로 시작하면 (startswith)
#        → handle_command(user_input, persona) 호출
#        → 반환된 (페르소나, 종료여부) 로 persona 갱신
#        → 종료여부가 True 이면 break
#   4) 그 외(일반 질문)
#        → ask(persona, user_input) 호출
#        → "[<persona>] <답변>" 형식으로 출력 + 빈 줄
# ------------------------------------------------------------

## 5-3. 챗봇 실행!


> 종료하려면 `/quit` 또는 `/exit` 입력.


In [ ]:
# [문제 13] 챗봇 실행
# ------------------------------------------------------------
# 요구사항
#   - run_chatbot() 을 기본값(분석가)으로 호출하세요.
#   - 실행 후 Step 6 시나리오 1~5를 직접 입력해 모두 통과시키세요.
#   - 종료: /quit 또는 /exit
# ------------------------------------------------------------